# [Harness name]: [one-line headline finding]

**Commit SHA:** `<40-character output of git rev-parse HEAD>`
**Repo:** <GitHub URL> (Apache-2.0 / MIT)
**model_api in submission.yaml:** `google-cloud/<strongest model>`
**Judge used for dev scores:** `<frontier judge or JUDGE_MODEL name>`

> **How to use this notebook.** Every `TODO` is yours to fill with real numbers from `eval/run.py` runs.
> Figure cells fall back to clearly labelled PLACEHOLDER data so the notebook runs end to end. Replace it
> with `results/dev_runs.csv` before submitting. Delete these instruction blocks, then run the word-count
> cell at the bottom. The Writeup must stay under 2,000 words.

## Headline finding (put your single strongest result here)

TODO: two or three sentences for an ML researcher with no engineering background.
Template: *"On the 29 dev tasks, adding [X] raised the mean score from A to B on [model], at C% more tokens.
The dominant failure was [mode], not [what people expect]."*

| Model | Dev score | Mean cost / task | Notes |
|---|---|---|---|
| TODO | TODO | TODO | TODO |

In [ ]:
import json, os, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

RESULTS = Path("results/dev_runs.csv")   # one row per (task, model, config) run
# Expected columns:
#   task_id, discipline, model, config, score, input_tokens, output_tokens,
#   cost_usd, runtime_s, n_calls, n_verify_attempts, failure_mode
# failure_mode is "none" for passes, otherwise one of FAILURE_MODES below.

FAILURE_MODES = [
    "misread drawing/callout",
    "wrong code clause/table",
    "unit/conversion error",
    "stopped early / missing deliverable",
    "tool/format error (IFC, xlsx, PDF)",
    "geometry/clash error",
    "arithmetic/solver error",
    "timeout/budget",
]

if RESULTS.exists():
    df = pd.read_csv(RESULTS)
    PLACEHOLDER = False
else:
    PLACEHOLDER = True
    rng = np.random.default_rng(0)
    rows = []
    for m in ["model-A", "model-B"]:
        for t in range(29):
            rows.append(dict(
                task_id=f"dev_{t:02d}",
                discipline=rng.choice(["structural", "civil", "electrical", "mechanical"]),
                model=m, config="final", score=float(rng.uniform(0, 1)),
                input_tokens=int(rng.integers(2e5, 3e6)), output_tokens=int(rng.integers(2e4, 3e5)),
                cost_usd=float(rng.uniform(0.2, 6)), runtime_s=float(rng.uniform(300, 9000)),
                n_calls=int(rng.integers(20, 300)), n_verify_attempts=int(rng.integers(0, 4)),
                failure_mode=str(rng.choice(["none"] + FAILURE_MODES))))
    df = pd.DataFrame(rows)

TAG = "PLACEHOLDER DATA - replace with real runs" if PLACEHOLDER else ""
print("Loaded", len(df), "rows;", "PLACEHOLDER" if PLACEHOLDER else "real data")

## 1. Insight and failure analysis (25 pts)

**The non-obvious lesson.** TODO: one claim a reader would not have guessed (for example "most failures were
not wrong physics but a missing deliverable the agent never noticed it owed").

### Controlled ablations
Each row changes one thing on the same 29 dev tasks and the same model.

| Change | Score before | Score after | Token cost change | Verdict |
|---|---|---|---|---|
| Add deliverable-checklist gate before `done` | TODO | TODO | TODO | TODO |
| Add unit-audit pass over every calculation | TODO | TODO | TODO | TODO |
| Re-read cited code table before using a value | TODO | TODO | TODO | TODO |
| Rasterize drawing sheets, read callouts visually | TODO | TODO | TODO | TODO |
| Context compaction every N turns | TODO | TODO | TODO | TODO |

### Ideas that failed
TODO: two or three things you tried that did not help, with a one-line reason each. Judges reward these.

### Where the dev tasks fail
Figure 3 (below) breaks the failures down by mode. TODO: describe the top one or two with a concrete example
(task id, what the agent read or computed, what the correct value was).

In [ ]:
# FIGURE 3: failure breakdown on the 29 dev tasks (final config, strongest model)
MODEL = df["model"].iloc[0] if PLACEHOLDER else df.groupby("model")["score"].mean().idxmax()
sub = df[(df.model == MODEL)]
if "config" in sub and (sub.config == "final").any():
    sub = sub[sub.config == "final"]
fails = sub[sub.failure_mode != "none"]
counts = fails.failure_mode.value_counts().reindex(FAILURE_MODES).fillna(0)
by_disc = fails.pivot_table(index="failure_mode", columns="discipline", values="task_id",
                            aggfunc="count").reindex(FAILURE_MODES).fillna(0)

fig, ax = plt.subplots(figsize=(8, 4.2))
by_disc.plot(kind="barh", stacked=True, ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Number of dev tasks (of 29)")
ax.set_ylabel("")
ax.set_title(f"Figure 3. Failure breakdown, {MODEL}" + (f"\n[{TAG}]" if TAG else ""))
plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/fig3_failures.png", dpi=200)
plt.show()

## 2. Engineering reasoning: one task, agent vs. licensed engineer (25 pts)

Pick one dev task that exercises a real skill (reading drawings, code tables, running a solver).

**Task:** TODO task id and one-sentence plain-language description (for example "size a spread footing on
clay and report settlement").

| Step | Licensed engineer's workflow | Our agent's trajectory | Held up? |
|---|---|---|---|
| 1. Gather inputs | TODO (read boring logs, loads, site constraints) | TODO | TODO |
| 2. Pick governing code/clause | TODO | TODO | TODO |
| 3. Calculate | TODO (with units tracked) | TODO | TODO |
| 4. Check against limits | TODO | TODO | TODO |
| 5. Produce stamped-quality deliverable | TODO | TODO | TODO |

**Where the agent took a shortcut:** TODO.
**Skills that separated passes from failures:** TODO (for example PDF and CAD callout reading, table lookup
fidelity, solver use, IFC validity checking).
**Jargon note for ML readers:** TODO. Define any term (IFC4, shear wall, settlement memo) in one clause on
first use.

## 3. Cross-model generalizability (15 pts)

All runs on the 29 public dev tasks. Use at least two sponsored models, ideally from different families.

In [ ]:
# Comparison table: score and cost per model (final config)
final = df[df.config == "final"] if (df.config == "final").any() else df
tbl = (final.groupby("model")
       .agg(mean_score=("score", "mean"),
            mean_cost_usd=("cost_usd", "mean"),
            mean_tokens=("input_tokens", lambda s: s.mean()),
            mean_runtime_min=("runtime_s", lambda s: s.mean() / 60),
            tasks=("task_id", "nunique"))
       .round(3).sort_values("mean_score", ascending=False))
if PLACEHOLDER:
    print("*** PLACEHOLDER DATA ***")
tbl

**What helped every model:** TODO.
**What helped only one model (and why you think so):** TODO.
**How model choice shifts accuracy vs. token cost:** TODO.
**Model committed in `submission.yaml`:** TODO, with the one-sentence reason (score, cost, or Pareto position).

## 4. Efficiency and scaling (20 pts)

Stage 2 ranks on the score-vs-cost Pareto frontier, so show where extra compute stops paying.

In [ ]:
# FIGURE 2: score vs. compute. Use whichever x-axis you varied:
# cost_usd, input_tokens, n_calls, runtime_s, or n_verify_attempts.
XCOL, XLABEL = "cost_usd", "Mean cost per task (USD)"

g = (df.groupby(["model", "config"])
       .agg(score=("score", "mean"), x=(XCOL, "mean")).reset_index())

fig, ax = plt.subplots(figsize=(7, 4.5))
for m, d in g.groupby("model"):
    d = d.sort_values("x")
    ax.plot(d.x, d.score, marker="o", label=m)
    for _, r in d.iterrows():
        ax.annotate(r.config, (r.x, r.score), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel(XLABEL)
ax.set_ylabel("Mean dev score (0-1)")
ax.set_title("Figure 2. Score vs. compute" + (f"\n[{TAG}]" if TAG else ""))
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/fig2_score_vs_cost.png", dpi=200)
plt.show()

**Where accuracy plateaus:** TODO (for example "no gain beyond 2 verification attempts").
**Estimated remaining headroom:** TODO, with your reasoning (for example share of failures that are
unrecoverable by more compute, such as misread drawings, vs. recoverable, such as stopping early).
**Stage 2 safety margin:** TODO. Note typical per-task runtime vs. the 6-hour limit, and what the harness
does as the budget runs out (it must still emit `done`; a timeout scores 0.0).

## 5. Architecture and clarity (15 pts)

Figure 1 shows the harness. TODO: three to five sentences on the design for a non-engineer, with the headline
finding already stated at the top.

In [ ]:
# FIGURE 1: architecture diagram. Edit the labels to match your actual harness.
boxes = {
    "Project folder\n(PDF sheets, CAD/IFC, specs,\nsoil logs, task prompt)": (0.02, 0.40),
    "Planner +\ncontext manager\n(deliverable checklist)": (0.26, 0.40),
    "Tool layer\nPDF/CAD reader, code-table lookup,\nsolver, xlsx/IFC writers": (0.52, 0.40),
    "Verification loop\nunit audit, clause recheck,\ngeometry/clash check": (0.52, 0.05),
    "Artifacts\n+ done frame": (0.80, 0.40),
}
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axis("off")
centers = {}
for label, (x, y) in boxes.items():
    w, h = 0.19, 0.30
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.01", fill=False, lw=1.5))
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=8)
    centers[label] = (x, y, w, h)

keys = list(boxes)
def arrow(a, b, rad=0.0):
    xa, ya, wa, ha = centers[a]; xb, yb, wb, hb = centers[b]
    if abs(xa - xb) < 1e-6:   # vertical
        p0 = (xa + wa / 2, ya + (0 if yb < ya else ha)); p1 = (xb + wb / 2, yb + (hb if yb < ya else 0))
    else:
        p0 = (xa + wa, ya + ha / 2); p1 = (xb, yb + hb / 2)
    ax.annotate("", xy=p1, xytext=p0,
                arrowprops=dict(arrowstyle="->", lw=1.3, connectionstyle=f"arc3,rad={rad}"))

arrow(keys[0], keys[1]); arrow(keys[1], keys[2]); arrow(keys[2], keys[4])
arrow(keys[2], keys[3], 0.0); arrow(keys[3], keys[1], 0.35)
ax.set_xlim(0, 1); ax.set_ylim(0, 0.8)
ax.set_title("Figure 1. Harness architecture (edit to match your implementation)")
plt.tight_layout()
plt.savefig("figures/fig1_architecture.png", dpi=200)
plt.show()

## Reproducibility and judge disclosure

- Dev artifacts generated with `uv run python eval/run.py`; `artifacts/manifest.json` hash matches.
- `uv run python -m kit check` and `uv run python -m kit artifacts` both pass at the commit above.
- Local judge used for dev scoring: TODO (name the model if it is not a frontier judge). Organizers re-score
  with the official judge, so expect some shift.
- No dev answers are hard-coded.

In [ ]:
# Word-count check for the final Writeup (markdown only; excludes code cells).
import json
nb = json.load(open("PLE_writeup.ipynb"))
text = "\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "markdown")
text = re.sub(r"[|>*#`-]", " ", text)
print("Approx. words:", len(text.split()), "(limit 2000; delete instruction text first)")